# Satang The Value - ONNX TTS Inference Runner (Google Colab)
Notebook นี้ใช้สำหรับ**ทดสอบและสร้างเสียงจากโมเดล ONNX โดยเฉพาะ** โดยไม่ต้องใช้ GPU (รันบน CPU Colab ได้) และอ่าน/บันทึกไฟล์ตรงกับ Google Drive

## 1. Mount Google Drive & ติดตั้ง Dependencies แบบไว (< 20 วินาที)

In [ ]:
# 1.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 1.2 ติดตั้ง uv (ใช้เวลา ~1 วินาที)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ["LOG_LEVEL"] = "DEBUG"

# 1.3 Clone หรือ Pull โค้ดล่าสุด
!git clone https://github.com/SatangThevalue/satangthevalue-tts-custom.git /content/satangthevalue-tts-custom 2>/dev/null || (cd /content/satangthevalue-tts-custom && git pull)
%cd /content/satangthevalue-tts-custom

# 1.4 ติดตั้งเฉพาะแพ็กเกจ Runtime (เบามาก ไม่ต้องลง PyTorch/CUDA)
!uv pip install --system onnxruntime pythainlp soundfile pedalboard pyyaml

## 2. โหลดโมเดล ONNX จาก Google Drive

In [ ]:
import os
from src.inference.engine import TTSEngine

# เลือกรุ่น INT8 (เร็วสุด ขนาด < 250MB) หรือรุ่น model.onnx ปกติ
model_int8 = "/content/drive/MyDrive/tts-project/04_onnx_exports/model_quant_int8.onnx"
model_fp16 = "/content/drive/MyDrive/tts-project/04_onnx_exports/model.onnx"

target_model = model_int8 if os.path.exists(model_int8) else model_fp16
print(f"กำลังโหลดโมเดล: {target_model}")

engine = TTSEngine(target_model)
print("โหลดโมเดล ONNX เข้าหน่วยความจำสำเร็จ พร้อมใช้งาน!")

## 3. พิมพ์ข้อความและสังเคราะห์เสียง (ฟังผลทันที)

In [ ]:
from IPython.display import Audio, display

# ป้อนข้อความภาษาไทยที่ต้องการให้ AI พูด (รองรับตัวเลข ตัวย่อภาษาอังกฤษ และคำทับศัพท์)
custom_text = "สวัสดีครับ นี่คือระบบเสียงสังเคราะห์ภาษาไทยที่รันผ่านระบบ ออนนิกซ์ จากกูเกิลไดรฟ์ เสียงชัดเจน นุ่มนวล และอ่านตัวเลขได้ถูกต้อง เช่น วันที่ 10 เดือน 5 เวลา 14 นาฬิกา 30 นาที"

# บันทึกผลลัพธ์ตรงเข้า Google Drive
output_file = "/content/drive/MyDrive/tts-project/04_onnx_exports/my_test_voice.wav"

# สังเคราะห์เสียงพร้อมผ่าน Spotify Pedalboard Studio DSP
res = engine.synthesize(
    text=custom_text,
    ref_audio_path=None,         # ปล่อย None สำหรับโทนเสียงปกติ
    output_wav_path=output_file,
    enable_mastering=True        # ปรับแต่งย่านความถี่และ Loudness มาตรฐานสตูดิโอ
)

print(f"✓ หน่วยเสียง (Phonemes): {res['phonemes']}")
print(f"✓ ความยาวเสียงที่ได้: {res['duration_sec']:.2f} วินาที")
print(f"✓ เวลาที่ใช้สร้าง: {res['elapsed_sec']:.2f} วินาที (RTF: {res['rtf']:.3f})")
print(f"✓ บันทึกไฟล์ที่: {res['output_path']}")

# เล่นเสียงผ่านเครื่องเล่นในหน้า Colab ทันที
display(Audio(res['output_path']))

## 4. ทดสอบคุมอารมณ์ด้วย In-Context Reference Audio (3-5 วินาที)
ระบบจะดึงจังหวะและไดนามิกของไฟล์เสียงตัวอย่างที่เลือกมาใช้กับประโยคใหม่

In [ ]:
import glob

# ตัวเลือก: ค้นหาไฟล์เสียงตัวอย่างจากโฟลเดอร์ 02_processed ใน Drive
wav_samples = sorted(glob.glob("/content/drive/MyDrive/tts-project/02_processed/wavs/*.wav"))
ref_clip = wav_samples[0] if wav_samples else None

if ref_clip:
    print(f"ใช้ไฟล์เสียงอ้างอิงอารมณ์: {ref_clip}")
    excited_text = "ข่าวด่วนวันนี้ ตลาดหุ้นและคริปโตปรับตัวสูงขึ้นอย่างรวดเร็ว ทำสถิติใหม่สูงสุดในรอบปี!"
    output_excited = "/content/drive/MyDrive/tts-project/04_onnx_exports/test_excited.wav"

    res_excited = engine.synthesize(
        text=excited_text,
        ref_audio_path=ref_clip,
        output_wav_path=output_excited,
        enable_mastering=True
    )
    display(Audio(res_excited['output_path']))
else:
    print("ยังไม่พบคลิปเสียงใน 02_processed/wavs คุณสามารถใส่พาธไฟล์ .wav ที่ต้องการตรงๆ ได้")